Install Library

In [ ]:
!pip install sentence-transformers faiss-cpu langchain-text-splitters beautifulsoup4 requests easyocr langchain-community gradio

Ambil links STT-NF

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

base_url = "https://akademik.nurulfikri.ac.id/"
html = requests.get(base_url).text
soup = BeautifulSoup(html, "html.parser")

links = set()
for a in soup.find_all("a", href=True):
    href = urljoin(base_url, a["href"])
    if (
        href.startswith(base_url)
        and "/category/" not in href
        and "wiki_action" not in href
    ):
        links.add(href)

print("Jumlah halaman:", len(links))

Jumlah halaman: 47


Scraping + Cleaning Teks Web

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

documents = []
noise_patterns = ["Search For", "Search", "Add Article", "Edit", "Login"]

for url in sorted(links):
    try:
        print("Scraping:", url)
        html = requests.get(url, timeout=15).text
        soup = BeautifulSoup(html, "html.parser")

        content = (
            soup.find("div", class_="entry-content")
            or soup.find("article")
            or soup.find("main")
        )
        if not content:
            continue

        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else "Tanpa Judul"

        text = " ".join(content.stripped_strings)
        for pattern in noise_patterns:
            text = re.sub(pattern, "", text)
        text = re.sub(r'\s+', ' ', text).strip()

        if len(text) < 100:
            continue

        documents.append({"title": title, "source": url, "content": text})

    except Exception as e:
        print("ERROR:", url, e)

df = pd.DataFrame(documents)
print("Jumlah halaman berhasil di-scrape:", len(df))

Scraping: https://akademik.nurulfikri.ac.id/1-daftar-sarpras/
Scraping: https://akademik.nurulfikri.ac.id/1-daftar-ulang/
Scraping: https://akademik.nurulfikri.ac.id/1-pedoman-mbkm/
Scraping: https://akademik.nurulfikri.ac.id/1-pedoman-tugas-akhir/
Scraping: https://akademik.nurulfikri.ac.id/1-penelitian-dan-pkm/
Scraping: https://akademik.nurulfikri.ac.id/1-satuan-kredit-semester-sks/
Scraping: https://akademik.nurulfikri.ac.id/1-sejarah/
Scraping: https://akademik.nurulfikri.ac.id/1-sistem-informasi/
Scraping: https://akademik.nurulfikri.ac.id/1-syarat-kelulusan/
Scraping: https://akademik.nurulfikri.ac.id/1-tentang-organisasi-kemahasiswaan/
Scraping: https://akademik.nurulfikri.ac.id/1-website-kampus/
Scraping: https://akademik.nurulfikri.ac.id/2-administrasi-keuangan/
Scraping: https://akademik.nurulfikri.ac.id/2-administrasi/
Scraping: https://akademik.nurulfikri.ac.id/2-aturan/
Scraping: https://akademik.nurulfikri.ac.id/2-laboratorium/
Scraping: https://akademik.nurulfikri.ac.id

OCR gambar dari Drive

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time

# Daftar link pilihanmu
urls = [
    "https://akademik.nurulfikri.ac.id/2-administrasi-keuangan/",
    "https://akademik.nurulfikri.ac.id/3-baak/",
    "https://akademik.nurulfikri.ac.id/4-upt-perpustakaan/",
    "https://akademik.nurulfikri.ac.id/3-syarat-pengambilan-skl-transkrip-dan-ijazah/",
    "https://akademik.nurulfikri.ac.id/4-wisuda/",
    "https://akademik.nurulfikri.ac.id/1-pedoman-mbkm/",
    "https://akademik.nurulfikri.ac.id/5-mars-stt-terpadu-nurul-fikri/",
    "https://akademik.nurulfikri.ac.id/6-hymne-stt-terpadu-nurul-fikri/",
    "https://akademik.nurulfikri.ac.id/2-laboratorium/",
    "https://akademik.nurulfikri.ac.id/5-unit-pendukung-kemahasiswaan/",
    "https://akademik.nurulfikri.ac.id/5-kalender-akademik/",
    "https://akademik.nurulfikri.ac.id/1-daftar-sarpras/",
    "https://akademik.nurulfikri.ac.id/2-sop-peminjaman-ruangan/",
    "https://akademik.nurulfikri.ac.id/3-struktur-organisasi/",
    "https://akademik.nurulfikri.ac.id/5-mitra/"
]

headers = {'User-Agent': 'Mozilla/5.0'}
output_folder = "poster_akademik"
os.makedirs(output_folder, exist_ok=True)
global_counter = 0

# New list to store metadata of downloaded images
downloaded_image_metadata = []

for page_url in urls:
    print(f"Mengakses: {page_url}")
    try:
        response = requests.get(page_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        # FOKUS: Cari konten di dalam div wkp-thecontent
        content = soup.find('div', class_='wkp-thecontent')

        if content:
            images = content.find_all('img')
            for img in images:
                # Ambil link gambar (data-src untuk lazy load)
                img_url = img.get('data-src') or img.get('src')

                # Filter: Buang yang mengandung kata logo, icon, atau svg
                if img_url and 'http' in img_url and not any(w in img_url for w in ['logo', 'icon', 'svg']):
                    # Handle broken image links (e.g., 'dev-akademik.nurulfikri.ac.id')
                    if 'dev-akademik.nurulfikri.ac.id' in img_url:
                        img_url = img_url.replace('dev-akademik.nurulfikri.ac.id', 'akademik.nurulfikri.ac.id')

                    try:
                        img_data = requests.get(img_url, headers=headers, timeout=5).content

                        # Filter ukuran: Hanya simpan gambar di atas 30KB (asumsi poster)
                        if len(img_data) > 30000:
                            filename = f"poster_{global_counter}.jpg"
                            filepath = os.path.join(output_folder, filename)
                            with open(filepath, 'wb') as f:
                                f.write(img_data)
                            print(f"  -> Berhasil simpan: {filepath}")

                            # Store metadata for downloaded images
                            downloaded_image_metadata.append({
                                'downloaded_filename': filename,
                                'original_image_url': img_url,
                                'page_url': page_url
                            })
                            global_counter += 1
                    except requests.exceptions.RequestException as e:
                        print(f"  -> Gagal mengunduh gambar dari {img_url}: {e}")
                        continue # Skip to the next image
        else:
            print("  -> Tidak ditemukan konten artikel (mungkin perlu Selenium).")

        time.sleep(2) # Jeda agar tidak dianggap spam

    except Exception as e:
        print(f"  -> Error accessing {page_url}: {e}")

print(f"\nSelesai! Total poster terkumpul: {global_counter}")

Mengakses: https://akademik.nurulfikri.ac.id/2-administrasi-keuangan/
  -> Berhasil simpan: poster_akademik/poster_0.jpg
  -> Berhasil simpan: poster_akademik/poster_1.jpg
  -> Berhasil simpan: poster_akademik/poster_2.jpg
Mengakses: https://akademik.nurulfikri.ac.id/3-baak/
  -> Berhasil simpan: poster_akademik/poster_3.jpg
Mengakses: https://akademik.nurulfikri.ac.id/4-upt-perpustakaan/
  -> Berhasil simpan: poster_akademik/poster_4.jpg
Mengakses: https://akademik.nurulfikri.ac.id/3-syarat-pengambilan-skl-transkrip-dan-ijazah/
  -> Berhasil simpan: poster_akademik/poster_5.jpg
Mengakses: https://akademik.nurulfikri.ac.id/4-wisuda/
  -> Berhasil simpan: poster_akademik/poster_6.jpg
Mengakses: https://akademik.nurulfikri.ac.id/1-pedoman-mbkm/
  -> Berhasil simpan: poster_akademik/poster_7.jpg
  -> Berhasil simpan: poster_akademik/poster_8.jpg
Mengakses: https://akademik.nurulfikri.ac.id/5-mars-stt-terpadu-nurul-fikri/
  -> Berhasil simpan: poster_akademik/poster_9.jpg
Mengakses: https:

In [ ]:
# Install Tesseract OCR and pytesseract if not already installed
!sudo apt-get install tesseract-ocr
!pip install pytesseract

import os
import pytesseract
from PIL import Image
from langchain_core.documents import Document

# 1. Pastikan folder ada
folder_path = 'poster_akademik'
all_docs = []

# 2. Loop semua file di folder
for filename in os.listdir(folder_path):
    if filename.endswith(('.jpg', '.png', '.jpeg')):
        file_path = os.path.join(folder_path, filename)

        try:
            # Lakukan OCR
            text = pytesseract.image_to_string(Image.open(file_path))

            # Buat dokumen LangChain
            doc = Document(
                page_content=text,
                metadata={"source": filename}
            )
            all_docs.append(doc)
            print(f"Berhasil ekstrak: {filename}")

        except Exception as e:
            print(f"Gagal proses {filename}: {e}")

print(f"\nBerhasil mengekstrak {len(all_docs)} dokumen dari gambar OCR.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Berhasil ekstrak: poster_8.jpg
Berhasil ekstrak: poster_22.jpg
Berhasil ekstrak: poster_15.jpg
Berhasil ekstrak: poster_12.jpg
Berhasil ekstrak: poster_19.jpg
Berhasil ekstrak: poster_42.jpg
Berhasil ekstrak: poster_24.jpg
Berhasil ekstrak: poster_1.jpg
Berhasil ekstrak: poster_6.jpg
Berhasil ekstrak: poster_31.jpg
Berhasil ekstrak: poster_2.jpg
Berhasil ekstrak: poster_4.jpg
Berhasil ekstrak: poster_26.jpg
Berhasil ekstrak: poster_33.jpg
Berhasil ekstrak: poster_16.jpg
Berhasil ekstrak: poster_3.jpg
Berhasil ekstrak: poster_32.jpg
Berhasil ekstrak: poster_25.jpg
Berhasil ekstrak: poster_7.jpg
Berhasil ekstrak: poster_29.jpg
Berhasil ekstrak: poster_20.jpg
Berhasil ekstrak: poster_39.jpg
Berhasil ekstrak: poster_28.jpg
Berhasil ekstrak: poster_38.jpg
Berhasil ekstra

In [ ]:
print("Cleaning OCR logic moved to cell k-Z-W4N7PLTW.")

Cleaning OCR logic moved to cell k-Z-W4N7PLTW.


Merge OCR + teks → save CSV

In [ ]:
import pandas as pd
import re

# Pastikan `df` (dari web scraping) tersedia
# df comes from c2vlDIPR_5rT

df_merged = df.copy() # Pastikan df_merged selalu didefinisikan dari df
df_merged['aggregated_ocr_result'] = ""
df_merged['content'] = df_merged['content'].fillna("").astype(str)

def bersihkan_dan_gabung(row):
    teks_web = row['content'].strip()
    teks_web = re.sub(r'Search For|Search|Add Article|Edit|Login', '', teks_web)
    teks_web = re.sub(r'\s+', ' ', teks_web).strip()
    return teks_web

df_merged['full_content'] = df_merged.apply(bersihkan_dan_gabung, axis=1)

# Hapus semua logika terkait OCR di sini, karena all_docs akan ditangani di k-Z-W4N7PLTW

df_merged.to_csv("dataset_nf_full.csv", index=False, encoding="utf-8-sig")
print("Total dokumen (web-scraped saja):", len(df_merged))
pd.set_option('display.max_colwidth', None)
display(df_merged[['full_content']].head())

Total dokumen (web-scraped saja): 41


,full_content
0,"1. Daftar Sarpras Gedung Perkuliahan Gedung perkuliahan merupakan salah satu fasilitas utama di sebuah kampus yang digunakan untuk mengadakan kuliah, seminar, dan berbagai aktivitas akademik. Berikut ini, gambar gedung perkuliahan kampus A dan kampus B, sebagai berikut: Gedung Kampus A STT-NF Gedung Kampus B STT-NF Perpustakaan STT-NF menyediakan fasilitas ruang perpustakaan untuk membantu mahasiswa dalam menambah wawasan pada bidang iptek dan ilmu terapan lainnya. Laboratorium STT-NF menyediakan fasilitas ruangan laboratorium yang digunakan untuk melakukan praktikum, seperti praktikum komputer, praktikum desain grafis dan lain sebagainya. Mahasiswa dapat melakukan praktikum di laboratorium untuk memahami konsep-konsep teori yang diajarkan dalam kuliah. Ruangan Laboratorium Kampus B Ruangan Laboratorium Kampus A Fasilitas Olahraga STT-NF menyediakan fasilitas olahraga yang bisa dimanfaatkan oleh mahasiswa STT-NF untuk tempat berkumpul, lomba olahraga atau mengadakan event yang sesuai dengan bakat di area kampus STT-NF. Lapangan Olahraga Lapangan Olahraga Tampak Dari Atas Auditorium STT-NF menyediakan fasilitas ruangan auditorium yang cukup representative , nyaman, dan bersih dengan kapasitas berjumlah kurang lebih 200 orang, juga dilengkapi dengan pendingin ruangan (AC), soundsystem, meja, kursi, karpet serta LCD/Screen projector sehingga mahasiswa/i yang ingin melakukan kegiatan perkuliahan, seminar, maupun penelitian yang ingin dipresentasikan ke Dosen dapat terealisasi dengan baik dan lancar. Ruangan Auditorium Ruang Rapat STT-NF menyediakan fasilitas ruang rapat yang digunakan untuk berbagai tujuan, seperti pertemuan bisnis, diskusi proyek, presentasi, pelatihan karyawan, rapat dewan direksi, atau pertemuan akademik. Ruang rapat biasanya dilengkapi dengan peralatan audio-visual, sistem suara, dan perangkat lunak presentasi. Ruang Rapat Kampus A Ruang Rapat Kampus B Mushola STT-NF menyediakan fasilitas musholla yang berada di kampus B STT-NF yang digunakan untuk melaksanakan shalat lima waktu sehari-hari dan melakukan ibadah lainnya, seperti bacaan Al-Qur’an, dzikir, dan berdoa. Mushola Kampus B Shaf Laki-Laki Mushola Kampus B Shaf Perempuan Kafetaria atau Kantin STT-NF menyediakan fasilitas kafetaria atau kantin digunakan untuk makan siang, makan ringan, atau minum kopi selama istirahat. Fasilitas kafetaria memiliki meja dan kursi yang nyaman bagi para pengunjung untuk duduk dan menikmati makanan. Kafetaria atau Kantin Kampus A Kafetaria atau Kantin Kampus B Ruang Tunggu STT-NF menyediakan fasilitas ruang tunggu yang berada di Kampus B STT-NF, yang dirancang untuk menerima tamu, pengunjung, atau klien. Ruang ini menciptakan lingkungan yang nyaman dan ramah untuk tamu yang menunggu untuk ditemui oleh seseorang atau menunggu pelaksanaan suatu layanan. Ruang Tunggu Kampus B STT-NF Fasilitas Parkir STT-NF menyediakan fasilitas parkir yang digunakan khusus untuk kendaraan bermotor seperti mobil, sepeda motor, dan sepeda agar dapat diparkir dengan aman dan tertib. Parkir Kampus A STT-NF Parkir Kampus B STT-NF Rooftop STT-NF menyediakan fasilitas Rooftop yang biasanya digunakan sebagai tempat yang tenang untuk membaca, berbicara, atau sekadar menikmati pemandangan kampus. Selain itu, Rooftop dapat menjadi tempat yang indah untuk mengadakan acara pertemuan alumni atau penerimaan mahasiswa baru. Rooftop Kampus A STT-NF yang terhubung dengan Perpustakaan Last Modified: January 22, 2026"
1,"1. Daftar Ulang Penjelasan Daftar ulang merupakan kegiatan registrasi administrasi dengan diawali pelunasan biaya BOP untuk semester berjalan. Pelunasan ini digunakan untuk membuka akses AIS untuk pengisian KRS. Mahasiswa yang tidak menjalankan proses registrasi administrasi akan menghadapi dampak berupa tidak bisa untuk mendaftar mata kuliah. Selain itu, mereka akan diberi status “tidak aktif” di semester tersebut, dan masa studi mereka tidak akan berlanjut, dengan tidak adanya biaya pendidikan yang harus dibayarkan. Sanksi Admin

Load CSV ke LangChain Documents

In [ ]:
import re
import pandas as pd # Import pandas
from langchain_core.documents import Document

def bersihkan_ocr(teks):
    # Buang karakter aneh, perbaiki spasi ganda
    teks = re.sub(r'[^\x00-\x7F]+', '', teks) # Hapus karakter non-ASCII
    teks = re.sub(r'[^\w\s.,?!-]', '', teks) # Buang karakter aneh
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

# Pastikan df_merged tersedia. Jika kernel reset, muat dari CSV.
if 'df_merged' not in globals():
    try:
        df_merged = pd.read_csv("dataset_nf_full.csv")
        print("df_merged loaded from CSV.")
    except FileNotFoundError:
        print("Error: dataset_nf_full.csv not found. Please run web scraping and merging cells (e52064cb) first.")
        df_merged = pd.DataFrame(columns=['title', 'source', 'full_content'])


# Mengubah setiap baris di df_merged menjadi Document object
documents = []
for index, row in df_merged.iterrows():
    # Menggabungkan semua informasi penting sebagai metadata
    doc = Document(
        page_content=row['full_content'],
        metadata={
            "title": row.get('title', 'N/A'),
            "source": row.get('source', 'N/A')
        }
    )
    documents.append(doc)

# Tambahkan dokumen dari OCR (all_docs) ke daftar dokumen utama
# all_docs diharapkan berasal dari XQWbe71gsPBk
if 'all_docs' in globals() and all_docs:
    documents.extend(all_docs)
    print(f"Berhasil menambahkan {len(all_docs)} dokumen OCR dari all_docs.")
else:
    print("Tidak ada dokumen OCR (all_docs) yang ditemukan untuk digabungkan.")


# Terapkan pembersihan OCR pada semua dokumen yang memiliki source 'poster_'
# (ini termasuk dokumen dari all_docs yang baru ditambahkan)
for doc in documents:
    if "poster_" in doc.metadata.get('source', '') : # Kondisi untuk filename poster
        doc.page_content = bersihkan_ocr(doc.page_content)

print(f"Total {len(documents)} dokumen siap untuk di-split.")

Berhasil menambahkan 43 dokumen OCR dari all_docs.
Total 84 dokumen siap untuk di-split.


Chunking & Embedding + Simpan Index FAISS

In [ ]:
!pip install langchain-huggingface

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # Import HuggingFaceEmbeddings
# Splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)

# Tambahkan prefix 'passage: ' (wajib untuk E5)
for chunk in chunks:
    chunk.page_content = "passage: " + chunk.page_content

# Inisialisasi embeddings (pastikan ini ada)
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")

# Buat FAISS Index
db_faiss = FAISS.from_documents(chunks, embeddings)
db_faiss.save_local("/content/faiss_index_akademik")
print("Database vektor final berhasil diperbarui dengan data Web & OCR!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Database vektor final berhasil diperbarui dengan data Web & OCR!


Setup LLM Groq & Fungsi Chatbot

In [ ]:
import os
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings # Import HuggingFaceEmbeddings

os.environ["GROQ_API_KEY"] = userdata.get('groq_api')
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

def chatbot(message, history=None):
    # 1. Pastikan selalu memuat versi terbaru dari disk
    embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
    db_faiss = FAISS.load_local(
        "/content/faiss_index_akademik",
        embeddings,
        allow_dangerous_deserialization=True
    )

    # 2. Sekarang gunakan retriever dari database yang baru dimuat
   # Menggunakan MMR agar pencarian lebih beragam (menangkap informasi dari berbagai sumber)
    retriever = db_faiss.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 15, "fetch_k": 30, "lambda_mult": 0.5} # Increased k and fetch_k
)

    # Tambahkan prefix 'query: ' ke pesan untuk retriever
    message_with_prefix = "query: " + message
    relevant_docs = retriever.invoke(message_with_prefix)

    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    prompt = f"""Kamu adalah asisten akademik STT Nurul Fikri. Berdasarkan informasi berikut, **identifikasi dan daftarkan semua program studi yang disebutkan**. Jika tidak ada informasi tentang program studi yang ditemukan, sebutkan bahwa informasi tidak tersedia.

{context}

Pertanyaan: {message}
Jawaban:"""
    response = llm.invoke(prompt)
    return response.content

Eksperimen & Evaluasi Respons Chatbot

In [ ]:
print(chatbot('apa saja syarat untuk mengambil SKL, Transkrip dan ijazah?'))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Berdasarkan informasi yang diberikan, syarat untuk mengambil SKL, Transkrip, dan Ijazah adalah sebagai berikut:

1. Sudah melakukan sidang tugas akhir STT-NF.
2. Menyerahkan pas photo berwarna, terbaru, dan berkualitas sangat baik dengan ukuran 4x6 cm sebanyak 6 lembar.
3. Menyerahkan bukti bebas pustaka yang dibuat oleh Bagian Perpustakaan.
4. Menyerahkan Surat Keterangan Bebas Keuangan yang dikeluarkan oleh Bagian Keuangan.
5. Sudah mengikuti wisuda.

Namun, perlu diingat bahwa informasi ini mungkin tidak lengkap atau tidak akurat, karena tidak ada informasi yang secara eksplisit menyebutkan syarat untuk mengambil SKL, Transkrip, dan Ijazah.

Sementara itu, program studi yang disebutkan dalam passage adalah:

* Program Sarjana STT Terpadu Nurul Fikri (tidak disebutkan nama program studi spesifik)
* Program studi yang terkait dengan mata kuliah seperti Pemrograman Web, Jaringan Komputer, Basis Data, dan lain-lain (mungkin terkait dengan program studi Teknik Informatika atau Ilmu Kompu

In [ ]:
# Cek apakah 3 prodi itu ada di 10 dokumen yang diambil
# Inisialisasi retriever di sini agar bisa digunakan secara independen
retriever = db_faiss.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 30, "lambda_mult": 0.5}
)

query = "query: apa saja program studi di stt nf?"
docs = retriever.invoke(query)

print(f"Total dokumen ditemukan: {len(docs)}")
for i, d in enumerate(docs):
    print(f"--- Dokumen {i+1} ---")
    print(d.page_content)

Total dokumen ditemukan: 10
--- Dokumen 1 ---
passage: akademik, ilmiah, dan sosial. Hasil penelitian ini dapat berupa publikasi ilmiah, temuan baru, dan kontribusi terhadap inovasi. Program ini hanya dapat dilakukan dan dilaksanakan oleh mahasiswa STT-NF semester 5 ke atas. Program ini bertujuan agar mahasiswa lulusan STT-NF dapat dan mampu membuat penelitian – penelitian yang baik ke depannya serta dapat memberikan bukti pengabdiannya kepada masyarakat. Kegiatan pengabdian kepada masyarakat dapat berupa pelatihan, konsultasi, penyuluhan, dan
--- Dokumen 2 ---
passage: 1. Sejarah Sejarah STT-NF diawali pada tahun 1994 dengan berdirinya lembaga Nurul Fikri Computer & Statistics (NCS), atau disebut juga sebagai NF Computer. Pada 1998, NF Computer mulai menyediakan pelatihan Linux dan aplikasi Open Source kepada masyarakat luas. Pada tahun 2000, NCS berubah nama menjadi Lembaga Pendidikan Komputer Nurul Fikri (LPK-NF), yang kemudian berganti nama kembali menjadi Lembaga Pendidikan dan Pe

In [ ]:
# Cek isi dokumen
web_docs = [d for d in documents if d.metadata.get('source') != 'ocr_poster'] # Sesuaikan source jika beda
print(f"Jumlah dokumen web: {len(web_docs)}")
if len(web_docs) > 0:
    print("Cuplikan teks web:", web_docs[0].page_content[:200])
else:
    print("WARNING: Tidak ada dokumen web ditemukan!")

Jumlah dokumen web: 84
Cuplikan teks web: 1. Daftar Sarpras Gedung Perkuliahan Gedung perkuliahan merupakan salah satu fasilitas utama di sebuah kampus yang digunakan untuk mengadakan kuliah, seminar, dan berbagai aktivitas akademik. Berikut 


In [ ]:
print(chatbot('apa saja program studi di stt nf'))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kw6a70g5fx2926rwjvycevj1` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99993, Requested 2279. Please try again in 32m43.008s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Cek isi dokumen
for doc in documents:
    if "Program Studi" in doc.page_content:
        print("Data ditemukan!")

In [ ]:
!pip install -q nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd

# Pertanyaan + referensi jawaban ideal (dari website asli)
eval_data = [
    {
        "pertanyaan": "Apa saja syarat untuk mengambil SKL?",
        "referensi": "Syarat pengambilan SKL antara lain telah menyelesaikan seluruh mata kuliah, tidak memiliki tanggungan administrasi keuangan, dan mengisi formulir permohonan SKL di BAAK."
    },
    {
        "pertanyaan": "Bagaimana cara mendaftar wisuda?",
        "referensi": "Pendaftaran wisuda dilakukan melalui BAAK dengan melengkapi persyaratan administrasi, bebas tanggungan keuangan, dan mengumpulkan berkas yang ditentukan."
    },
    {
        "pertanyaan": "Apa itu MBKM?",
        "referensi": "MBKM adalah program Merdeka Belajar Kampus Merdeka yang memberikan kesempatan mahasiswa belajar di luar program studi selama 1-2 semester."
    },
    {
        "pertanyaan": "Apa saja layanan perpustakaan STT-NF?",
        "referensi": "Perpustakaan STT-NF menyediakan layanan peminjaman buku, akses jurnal digital, ruang baca, dan layanan referensi untuk civitas akademika."
    },
    {
        "pertanyaan": "Apa syarat pengambilan ijazah?",
        "referensi": "Syarat pengambilan ijazah meliputi telah dinyatakan lulus wisuda, bebas tanggungan administrasi, dan mengumpulkan berkas persyaratan ke BAAK."
    },
]

smoother = SmoothingFunction().method1
hasil_evaluasi = []

print("=" * 70)
for item in eval_data:
    jawaban_chatbot = chatbot(item["pertanyaan"])

    # Hitung BLEU score
    referensi_token = [item["referensi"].lower().split()]
    hipotesis_token = jawaban_chatbot.lower().split()
    bleu = sentence_bleu(referensi_token, hipotesis_token, smoothing_function=smoother)

    # Relevansi manual: cek apakah keyword penting ada di jawaban
    keywords = item["referensi"].lower().split()[:5]
    keyword_match = sum(1 for k in keywords if k in jawaban_chatbot.lower())
    relevansi = "Relevan" if keyword_match >= 2 else "Kurang Relevan"

    print(f"Q: {item['pertanyaan']}")
    print(f"A: {jawaban_chatbot[:300]}")
    print(f"BLEU Score : {bleu:.4f}")
    print(f"Relevansi  : {relevansi}")
    print("=" * 70)

    hasil_evaluasi.append({
        "pertanyaan": item["pertanyaan"],
        "jawaban_chatbot": jawaban_chatbot,
        "referensi": item["referensi"],
        "bleu_score": round(bleu, 4),
        "relevansi": relevansi
    })

df_eval = pd.DataFrame(hasil_evaluasi)
print(f"\nRata-rata BLEU Score: {df_eval['bleu_score'].mean():.4f}")
print(f"Relevan: {(df_eval['relevansi'] == 'Relevan').sum()}/{len(df_eval)}")
df_eval.to_csv("hasil_evaluasi_chatbot.csv", index=False, encoding="utf-8-sig")
print("\nTersimpan di hasil_evaluasi_chatbot.csv")

Demo — Gradio UI

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chatbot,
    title="Chatbot Akademik STT-NF"
)

demo.launch()